# Train SafeDrive on Kaggle (GPU, no local dataset download)

This notebook attaches the FL3D dataset from Kaggle’s cloud storage, then fine-tunes ImageNet-pretrained MobileNetV3-Small. Only the resulting model and evaluation report need to be downloaded. The dataset stays in Kaggle.

## Before running
1. In Kaggle, create a **New Notebook** and add the public dataset **Frame Level Driver Drowsiness Detection (FL3D)** (`matjazmuc/frame-level-driver-drowsiness-detection-fl3d`) with **Add Input**.
2. In Notebook options, select a **GPU** accelerator and turn **Internet** on so the pretrained weights can be downloaded.
3. Push this project’s code to GitHub first. The setup cell clones its `main` branch.
4. Run all cells. The held-out test split is by source video. This is a research prototype; higher scores do not make it safe for driving use.


In [ ]:
import subprocess, torch, torchvision
assert torch.cuda.is_available(), 'Enable a GPU in Notebook options before training.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
REPO = '/kaggle/working/SafeDrive-Monitor'
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Pravith101/SafeDrive-Monitor.git', REPO], check=True)
%cd /kaggle/working/SafeDrive-Monitor


In [ ]:
from pathlib import Path
INPUT_ROOT = Path('/kaggle/input')
annotations = list(INPUT_ROOT.rglob('annotations_final.json'))
assert annotations, 'Dataset missing. Use Add Input to attach the FL3D Kaggle dataset.'
DATASET_DIR = annotations[0].parent
print('Found FL3D annotations:', len(annotations))
print('Dataset root:', DATASET_DIR)


In [ ]:
OUTPUT = '/kaggle/working/driver_state_cnn.pth'
subprocess.run([
    'python', 'models/driver_state_cnn.py',
    '--dataset-dir', str(INPUT_ROOT),
    '--pretrained', '--epochs', '12', '--batch-size', '64', '--patience', '4',
    '--output', OUTPUT,
], check=True)
print('Model:', OUTPUT)
print('Evaluation:', '/kaggle/working/driver_state_evaluation.json')


## Download the trained model

After the notebook finishes, choose **Save Version → Save & Run All**. When the run completes, download `driver_state_cnn.pth` from the notebook’s **Output** section and place it at `weights/driver_state_cnn.pth` in your project. You can also download `driver_state_evaluation.json` to review the held-out video results. The checkpoint includes the correct preprocessing settings for the webcam app.

Run the app locally with `python live_inference.py`.
